# MRW-CNN — Plant Disease Classification
**Paper:** Real-Time Plant Disease Dataset Development and Detection of Plant Disease Using Deep Learning  
**Authors:** Diana Susan Joseph, Pranav M. Pawar, Kaustubh Chakradeo  
**Journal:** IEEE Access, Vol. 12, 2024  
**DOI:** 10.1109/ACCESS.2024.3358333  

---
## Architecture Summary (MRW-CNN — Maize, Rice, Wheat CNN)
- Input: 224x224x3, normalize /255 (train from scratch — NOT ImageNet mean/std)
- 5 Convolution blocks, each = Conv -> BatchNorm -> ReLU (NO pooling between or after blocks)
- After conv blocks: Global Average Pooling -> Dense(num_classes, softmax)
- Train from scratch (no pretrained backbone)

> **CONFIRMED from paper Figure 9** (read directly via vision — see `inputs/paper_content.md` -> 'EXTRACTED FIGURES (visual analysis)'):
> - Exactly **5 conv blocks**, each = `Conv2D -> BatchNormalization -> ReLU` (in that order)
> - Per-block filters: **32 -> 64 -> 64 -> 256 -> 256** (blocks 2 and 3 both use 64; blocks 4 and 5 both use 256 — NOT a smooth doubling)
> - Kernel size: **3x3** for every block (explicitly labeled in each box of Fig. 9)
> - **NO MaxPooling / pooling layer of any kind** anywhere — only a single Global Average Pooling at the very end (confirmed absent from both Fig. 9 and Section V text)
> - Head: Global Average Pooling -> Dense(num_classes) -> Softmax

> **ASSUMPTION (the ONLY remaining assumption — NOT in paper, user-approved 2026-06-16):**
> - **Stride = 2** for all 5 conv layers, **padding = 'same'**. The paper annotates no stride/padding anywhere (text or figure). Since there is no pooling, stride=2 acts as the downsampling mechanism, reducing spatial size 224 -> 112 -> 56 -> 28 -> 14 -> 7 before Global Average Pooling. This is a standard pooling-free CNN pattern and lighter on compute than stride=1.

---
## Dataset Substitution Notice
The paper's original maize/rice/wheat dataset is **NOT publicly available**. This notebook reproduces the MRW-CNN **architecture (per Fig. 9) and hyperparameters exactly per the paper**, but runs on a **substitute dataset**: the user's chili pepper dataset (5 classes: `anthracnose`, `healthy`, `leaf_curl`, `leaf_spot`, `yellow_leaf`). Only the image source is swapped — the model and training config stay faithful to the paper.

---
## Experiment Design (project rules)
- **Scenario 1 (Baseline):** Resize 224x224 + /255 only. No augmentation.
- **Scenario 2 (Augmented):** Rotation + Zoom + Horizontal Flip + Brightness on training set. Val/Test never augmented.
- Metrics: Accuracy, F1-Macro, Recall-Macro, Total Parameters, Classification Report per class
- Best model checkpoint (val_accuracy) reloaded before test evaluation
- No EarlyStopping — full 100 epochs per paper

> Note: The paper used **offline** augmentation (generating new images saved to folders). This notebook uses **online** augmentation in the data pipeline per project rules — a methodological difference from the paper.

## Cell 1: Install Dependencies

In [ ]:
!pip install -q scikit-learn matplotlib numpy openpyxl Pillow

## Cell 2: Imports and Reproducibility Seeds

In [ ]:
import os
import gc
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.losses import CategoricalCrossentropy

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, recall_score, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility (CORE RULE #7)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

# GPU Detection (CORE RULE #8 — TF equivalent)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU detected — running on CPU")

# ── Multi-GPU: MirroredStrategy (Rule 8) ──────────────────────────────────────
strategy = tf.distribute.MirroredStrategy()
n_gpus = strategy.num_replicas_in_sync
print(f"[Multi-GPU] MirroredStrategy: {n_gpus} GPU(s) available")


## Cell 3: Configuration

In [ ]:
# Dataset Paths
# SUBSTITUTE DATASET: chili pepper (paper's maize/rice/wheat dataset is not public).
# Dataset is PRE-SPLIT into train/val/test — no splitting logic needed.
DATASET_ROOT = "/kaggle/input/datasets/tminhhi/dataset-ot/dataset_smartcrop_plit_300"
TRAIN_DIR    = os.path.join(DATASET_ROOT, "train")
VAL_DIR      = os.path.join(DATASET_ROOT, "val")
TEST_DIR     = os.path.join(DATASET_ROOT, "test")

# Auto-detect class names (CORE RULE #12)
class_names = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])
NUM_CLASSES = len(class_names)
print(f"Classes detected: {NUM_CLASSES}")
print(f"Class names: {class_names}")

# MRW-CNN Architecture Hyperparameters
IMAGE_SIZE = 224          # Paper Section III.B: images resized to 224x224 (CONFIRMED)

# CONFIRMED from paper Figure 9 (read via vision):
#   5 conv blocks; filters 32->64->64->256->256; kernel 3x3 each; NO pooling.
CONV_FILTERS = [32, 64, 64, 256, 256]   # CONFIRMED (Fig. 9)
KERNEL_SIZE  = (3, 3)                    # CONFIRMED (Fig. 9)

# ASSUMED (NOT in paper — user-approved 2026-06-16): only stride & padding.
# stride=2 acts as downsampling since there is NO pooling layer.
CONV_STRIDE  = (2, 2)                    # ASSUMED
CONV_PADDING = 'same'                    # ASSUMED
# NOTE: NO MaxPooling anywhere (CONFIRMED absent from Fig. 9 + Section V text).

# Training Hyperparameters (ALL CONFIRMED from paper Section VI.B)
OPTIMIZER_NAME = "SGD"    # Paper Section VI.B + V (CONFIRMED)
LEARNING_RATE  = 0.001    # Paper Section VI.B (CONFIRMED)
BATCH_SIZE     = 32       # Paper Section VI.B (CONFIRMED)
EPOCHS         = 100      # Paper Section VIII.A 'trained for 100 epochs' (CONFIRMED)
# Loss = CategoricalCrossentropy (user-confirmed; standard for multi-class + softmax)

# Output Paths
OUTPUT_DIR = "/kaggle/working"
MODEL_S1   = os.path.join(OUTPUT_DIR, "best_model_s1.keras")
MODEL_S2   = os.path.join(OUTPUT_DIR, "best_model_s2.keras")

LC_S1_PATH = os.path.join(OUTPUT_DIR, "learning_curve_s1.png")
CM_S1_PATH = os.path.join(OUTPUT_DIR, "confusion_matrix_s1.png")
LC_S2_PATH = os.path.join(OUTPUT_DIR, "learning_curve_s2.png")
CM_S2_PATH = os.path.join(OUTPUT_DIR, "confusion_matrix_s2.png")

print("\nConfiguration summary:")
print(f"  Image size:    {IMAGE_SIZE}x{IMAGE_SIZE}  (paper-confirmed)")
print(f"  Conv filters:  {CONV_FILTERS}  (CONFIRMED — Fig. 9)")
print(f"  Kernel size:   {KERNEL_SIZE}  (CONFIRMED — Fig. 9)")
print(f"  Pooling:       NONE  (CONFIRMED absent — Fig. 9 + text)")
print(f"  Stride:        {CONV_STRIDE}, padding={CONV_PADDING}  (ASSUMED — not in paper)")
print(f"  Optimizer:     {OPTIMIZER_NAME}  (paper-confirmed)")
print(f"  Learning rate: {LEARNING_RATE}  (paper-confirmed)")
print(f"  Batch size:    {BATCH_SIZE}  (paper-confirmed)")
print(f"  Epochs:        {EPOCHS}  (paper-confirmed)")

## Cell 4: Dataset Distribution and Class Imbalance Check (CORE RULE #11)

In [ ]:
# Count samples per class in train set
class_counts = {}
for cls in class_names:
    cls_path = os.path.join(TRAIN_DIR, cls)
    n = len([f for f in os.listdir(cls_path)
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))])
    class_counts[cls] = n

total_train = sum(class_counts.values())
max_count   = max(class_counts.values())
min_count   = min(class_counts.values())
imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')

print(f"Training set distribution ({total_train} total images):")
for cls, n in class_counts.items():
    print(f"  {cls}: {n} ({n/total_train*100:.1f}%)")

print(f"\nMax count: {max_count}  |  Min count: {min_count}")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")

# Bar chart
plt.figure(figsize=(max(8, NUM_CLASSES * 0.9), 5))
bars = plt.bar(class_names, [class_counts[c] for c in class_names],
               color='steelblue', edgecolor='white')
plt.title('Dataset Distribution — Training Set', fontsize=14, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right', fontsize=9)
for bar, cls in zip(bars, class_names):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             str(class_counts[cls]), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()
plt.close()

# Class weight decision
USE_CLASS_WEIGHTS = (imbalance_ratio >= 1.5)
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    print(f"\n[INFO] Imbalance ratio={imbalance_ratio:.2f} >= 1.5 — computing class weights.")
    y_train_labels = []
    for idx, cls in enumerate(class_names):
        y_train_labels.extend([idx] * class_counts[cls])
    y_train_labels = np.array(y_train_labels)
    cw = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(NUM_CLASSES),
        y=y_train_labels
    )
    class_weight_dict = {i: cw[i] for i in range(NUM_CLASSES)}
    print(f"Class weights: {class_weight_dict}")
else:
    print(f"\n[INFO] Imbalance ratio={imbalance_ratio:.2f} < 1.5 — no class weighting applied.")

## Cell 5: MRW-CNN Architecture Builder

In [ ]:
# ===========================================================
# MRW-CNN Architecture (Joseph et al., IEEE Access 2024)
# ===========================================================
# CONFIRMED from paper Figure 9 (read via vision) + Section V text:
#   - 5 Convolution blocks, each = Conv2D -> BatchNorm -> ReLU
#   - Per-block filters: 32 -> 64 -> 64 -> 256 -> 256
#   - Kernel 3x3 for every block
#   - NO pooling between/after conv blocks (only one GAP at the end)
#   - Head: Global Average Pooling -> Dense(num_classes, softmax)
#   - Train from scratch (no pretrained weights)
# ASSUMED (NOT in paper — user-approved): only stride=2 + padding='same'
#   (downsampling mechanism in the absence of pooling).

def build_mrw_cnn(num_classes, image_size, conv_filters, kernel_size,
                  conv_stride, conv_padding):
    """Full MRW-CNN model (Keras Functional API). No pooling between conv blocks."""
    inputs = keras.Input(shape=(image_size, image_size, 3), name='input')
    x = inputs

    for i, f in enumerate(conv_filters, start=1):
        x = layers.Conv2D(
            filters=f, kernel_size=kernel_size, strides=conv_stride,
            padding=conv_padding, use_bias=False, name=f'conv{i}'
        )(x)
        x = layers.BatchNormalization(name=f'bn{i}')(x)
        x = layers.Activation('relu', name=f'relu{i}')(x)
        # NO pooling layer — confirmed absent from Fig. 9 + Section V text.

    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='classifier')(x)

    return Model(inputs=inputs, outputs=outputs, name='MRW_CNN')


print("MRW-CNN architecture builder defined.")
print(f"  Conv blocks: {len(CONV_FILTERS)}  | filters={CONV_FILTERS} (CONFIRMED Fig. 9)")
print("  Each block: Conv -> BatchNorm -> ReLU  (NO pooling)")
print(f"  Stride={CONV_STRIDE}, padding={CONV_PADDING} (ASSUMED) — downsampling in place of pooling")
print("  Head: GlobalAveragePooling -> Dense(softmax)")

## Cell 6: Utility Functions (Evaluation, Visualization)

In [ ]:
def evaluate_model(model, test_generator, model_name='MRW-CNN'):
    """Run inference on test generator, return metrics dict + y_true/y_pred."""
    test_generator.reset()
    steps = len(test_generator)
    y_pred_probs = model.predict(test_generator, steps=steps, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)

    y_true = test_generator.classes[:len(y_pred)]

    acc       = accuracy_score(y_true, y_pred)
    f1_macro  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    total_p   = model.count_params()

    metrics = {
        'model_name':   model_name,
        'accuracy':     acc,
        'f1_macro':     f1_macro,
        'recall_macro': rec_macro,
        'total_params': total_p
    }
    return metrics, y_true, y_pred


def plot_confusion_matrix(y_true, y_pred, class_names, save_path, title='Confusion Matrix'):
    """Plot and save confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    n_classes = len(class_names)
    fig_size = max(8, n_classes * 0.8)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))

    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)

    tick_marks = np.arange(n_classes)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(class_names, fontsize=9)

    fmt = 'd'
    thresh = cm.max() / 2.0
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha='center', va='center', fontsize=9,
                    color='white' if cm[i, j] > thresh else 'black')

    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Confusion matrix saved: {save_path}")


def plot_learning_curve(history, save_path, title='Learning Curve'):
    """Plot and save training/validation accuracy and loss curves."""
    epochs_ran = len(history.history['accuracy'])
    epoch_range = range(1, epochs_ran + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epoch_range, history.history['accuracy'],     label='Train Acc', color='royalblue')
    axes[0].plot(epoch_range, history.history['val_accuracy'], label='Val Acc',   color='tomato', linestyle='--')
    axes[0].set_title('Accuracy', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epoch_range, history.history['loss'],     label='Train Loss', color='royalblue')
    axes[1].plot(epoch_range, history.history['val_loss'], label='Val Loss',   color='tomato', linestyle='--')
    axes[1].set_title('Loss', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Learning curve saved: {save_path}")


print("Utility functions defined.")

---
# SCENARIO 1 — BASELINE (No Augmentation)
Training transform: Resize 224x224 + /255 only

## Cell 7: Scenario 1 — Data Generators (Baseline)

In [ ]:
# Scenario 1: Baseline generators
# Normalization: /255 (user-confirmed — train from scratch, NOT ImageNet mean/std)
# Training: ONLY resize + /255, no augmentation (CORE RULE #3)
# Val / Test: same transform as training baseline

baseline_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_gen_s1 = baseline_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_gen_s1 = baseline_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen_s1 = baseline_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Update class_names from generator (authoritative order)
class_names = list(train_gen_s1.class_indices.keys())

print(f"Train samples:      {train_gen_s1.samples}")
print(f"Validation samples: {val_gen_s1.samples}")
print(f"Test samples:       {test_gen_s1.samples}")
print(f"Class index map:    {train_gen_s1.class_indices}")
print(f"Num classes:        {len(class_names)}")

## Cell 8: Scenario 1 — Build and Summarize Model (CORE RULE #5)

In [ ]:

# Build MRW-CNN model for Scenario 1 (inside strategy.scope() for multi-GPU)
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

with strategy.scope():
    model_s1 = build_mrw_cnn(
        num_classes  = NUM_CLASSES,
        image_size   = IMAGE_SIZE,
        conv_filters = CONV_FILTERS,
        kernel_size  = KERNEL_SIZE,
        conv_stride  = CONV_STRIDE,
        conv_padding = CONV_PADDING
    )

    model_s1.compile(
        optimizer=SGD(learning_rate=LEARNING_RATE),
        loss=CategoricalCrossentropy(),
        metrics=['accuracy']
    )

# Model summary (CORE RULE #5)
model_s1.summary()

total_params = model_s1.count_params()
print(f"\nTotal parameters: {total_params:,}  ({total_params/1e6:.2f}M)")
print("Note: Paper does NOT report a specific parameter count for MRW-CNN")
print("      (only states 'less number of layers'). No numeric target to compare.")


## Cell 9: Scenario 1 — Training

In [ ]:

# Callbacks (CORE RULE #10: best checkpoint by val_accuracy).
# No EarlyStopping — full 100 epochs per paper.
callbacks_s1 = [
    ModelCheckpoint(
        filepath=MODEL_S1,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# Train Scenario 1
print("=" * 60)
print("SCENARIO 1 — Baseline (No Augmentation)")
print("Training: Resize 224x224 + /255 ONLY")
print("=" * 60)

history_s1 = model_s1.fit(
    train_gen_s1,
    epochs=EPOCHS,
    steps_per_epoch=len(train_gen_s1),
    validation_data=val_gen_s1,
    validation_steps=len(val_gen_s1),
    callbacks=callbacks_s1,
    class_weight=class_weight_dict,
    verbose=1
)

print("\nScenario 1 training complete.")
print(f"Best model saved to: {MODEL_S1}")


## Cell 10: Scenario 1 — Learning Curve

In [ ]:
plot_learning_curve(
    history_s1,
    save_path=LC_S1_PATH,
    title='MRW-CNN — Scenario 1 (Baseline) Learning Curve'
)
BASELINE_LEARNING_CURVE_PATH = LC_S1_PATH

## Cell 11: Scenario 1 — Classification Report (CORE RULE #4)

In [ ]:
# Reload best checkpoint before test evaluation (CORE RULE #10)
print(f"Loading best model from: {MODEL_S1}")
model_s1 = keras.models.load_model(MODEL_S1, compile=False)
model_s1.compile(
    optimizer=SGD(learning_rate=LEARNING_RATE),
    loss=CategoricalCrossentropy(),
    metrics=['accuracy']
)

baseline_metrics, y_true_s1, y_pred_s1 = evaluate_model(
    model_s1, test_gen_s1, model_name='MRW-CNN'
)

print("\n" + "=" * 60)
print("SCENARIO 1 — TEST SET RESULTS (Best Checkpoint)")
print("=" * 60)
print(f"  Accuracy:      {baseline_metrics['accuracy']:.4f}  ({baseline_metrics['accuracy']*100:.2f}%)")
print(f"  F1-Macro:      {baseline_metrics['f1_macro']:.4f}")
print(f"  Recall-Macro:  {baseline_metrics['recall_macro']:.4f}")
print(f"  Total Params:  {baseline_metrics['total_params']:,}  ({baseline_metrics['total_params']/1e6:.2f}M)")

print("\n--- Classification Report (per class) ---")
print(classification_report(
    y_true_s1, y_pred_s1,
    target_names=class_names,
    digits=4,
    zero_division=0
))

## Cell 12: Scenario 1 — Confusion Matrix

In [ ]:
plot_confusion_matrix(
    y_true_s1, y_pred_s1,
    class_names=class_names,
    save_path=CM_S1_PATH,
    title='MRW-CNN — Scenario 1 (Baseline) Confusion Matrix'
)
BASELINE_CONFUSION_MATRIX_PATH = CM_S1_PATH

## Cell 13: Clear Session After Scenario 1 (CORE RULE #13)

In [ ]:
# Clear GPU memory before Scenario 2
del model_s1
tf.keras.backend.clear_session()
gc.collect()
print("Session cleared. Ready for Scenario 2.")

---
# SCENARIO 2 — AUGMENTED
Training: Rotation + Zoom + Horizontal Flip + Brightness  
Val/Test: same as Scenario 1 baseline (no augmentation)

## Cell 14: Scenario 2 — Data Generators (Augmented)

In [ ]:
# Scenario 2: Augmented training generator (CORE RULE #3)
# 4 mandatory augmentation techniques:
#   1. Rotation        (rotation_range=90 — paper used 90 degrees)
#   2. Zoom/Scale      (zoom_range=0.2)
#   3. Horizontal Flip (horizontal_flip=True)
#   4. Brightness      (brightness_range=[0.8, 1.2] — ±20% around original)
#      NOTE: [0.2, 0.8] would make ALL training images 20-80% darker than original,
#      causing train/test distribution mismatch and ~20% accuracy drop.
#      Correct range: symmetric around 1.0 so images can be brighter OR darker.
# Val/Test: no augmentation (same as Scenario 1 baseline)

aug_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=90,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)  # no aug for val/test

train_gen_s2 = aug_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_gen_s2 = eval_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen_s2 = eval_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Train samples (Scenario 2): {train_gen_s2.samples}")
print(f"Val samples:                {val_gen_s2.samples}")
print(f"Test samples:               {test_gen_s2.samples}")
print("Augmentation: rotation=90, zoom=0.2, h_flip=True, brightness=[0.8, 1.2]")

## Cell 15: Scenario 2 — Build and Summarize Model

In [ ]:

# Reinitialize model from scratch for Scenario 2 (inside strategy.scope())
tf.random.set_seed(SEED)

with strategy.scope():
    model_s2 = build_mrw_cnn(
        num_classes  = NUM_CLASSES,
        image_size   = IMAGE_SIZE,
        conv_filters = CONV_FILTERS,
        kernel_size  = KERNEL_SIZE,
        conv_stride  = CONV_STRIDE,
        conv_padding = CONV_PADDING
    )

    model_s2.compile(
        optimizer=SGD(learning_rate=LEARNING_RATE),
        loss=CategoricalCrossentropy(),
        metrics=['accuracy']
    )

model_s2.summary()
print(f"\nTotal parameters (S2): {model_s2.count_params():,}  ({model_s2.count_params()/1e6:.2f}M)")


## Cell 16: Scenario 2 — Training

In [ ]:

# Callbacks for Scenario 2 (no EarlyStopping — full 100 epochs)
callbacks_s2 = [
    ModelCheckpoint(
        filepath=MODEL_S2,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# Train Scenario 2
print("=" * 60)
print("SCENARIO 2 — Augmented")
print("Augmentation: Rotation=90, Zoom=0.2, H-Flip, Brightness=[0.8, 1.2]")
print("=" * 60)

history_s2 = model_s2.fit(
    train_gen_s2,
    epochs=EPOCHS,
    steps_per_epoch=len(train_gen_s2),
    validation_data=val_gen_s2,
    validation_steps=len(val_gen_s2),
    callbacks=callbacks_s2,
    class_weight=class_weight_dict,
    verbose=1
)

print("\nScenario 2 training complete.")
print(f"Best model saved to: {MODEL_S2}")


## Cell 17: Scenario 2 — Learning Curve

In [ ]:
plot_learning_curve(
    history_s2,
    save_path=LC_S2_PATH,
    title='MRW-CNN — Scenario 2 (Augmented) Learning Curve'
)
AUG_LEARNING_CURVE_PATH = LC_S2_PATH

## Cell 18: Scenario 2 — Classification Report

In [ ]:
# Reload best checkpoint before test evaluation (CORE RULE #10)
print(f"Loading best model from: {MODEL_S2}")
model_s2 = keras.models.load_model(MODEL_S2, compile=False)
model_s2.compile(
    optimizer=SGD(learning_rate=LEARNING_RATE),
    loss=CategoricalCrossentropy(),
    metrics=['accuracy']
)

aug_metrics, y_true_s2, y_pred_s2 = evaluate_model(
    model_s2, test_gen_s2, model_name='MRW-CNN'
)

print("\n" + "=" * 60)
print("SCENARIO 2 — TEST SET RESULTS (Best Checkpoint)")
print("=" * 60)
print(f"  Accuracy:      {aug_metrics['accuracy']:.4f}  ({aug_metrics['accuracy']*100:.2f}%)")
print(f"  F1-Macro:      {aug_metrics['f1_macro']:.4f}")
print(f"  Recall-Macro:  {aug_metrics['recall_macro']:.4f}")
print(f"  Total Params:  {aug_metrics['total_params']:,}  ({aug_metrics['total_params']/1e6:.2f}M)")

print("\n--- Classification Report (per class) ---")
print(classification_report(
    y_true_s2, y_pred_s2,
    target_names=class_names,
    digits=4,
    zero_division=0
))

## Cell 19: Scenario 2 — Confusion Matrix

In [ ]:
plot_confusion_matrix(
    y_true_s2, y_pred_s2,
    class_names=class_names,
    save_path=CM_S2_PATH,
    title='MRW-CNN — Scenario 2 (Augmented) Confusion Matrix'
)
AUG_CONFUSION_MATRIX_PATH = CM_S2_PATH

## Cell 20: Clear Session After Scenario 2

In [ ]:
del model_s2
tf.keras.backend.clear_session()
gc.collect()
print("Session cleared after Scenario 2.")

---
# RESULTS SUMMARY

## Cell 21: Results Comparison

In [ ]:
print("\n" + "=" * 70)
print("MRW-CNN — FINAL RESULTS COMPARISON")
print("=" * 70)
print(f"{'Metric':<20} {'Scenario 1 (Baseline)':>22} {'Scenario 2 (Augmented)':>22}")
print("-" * 70)
print(f"{'Accuracy':<20} {baseline_metrics['accuracy']*100:>21.2f}% {aug_metrics['accuracy']*100:>21.2f}%")
print(f"{'F1-Macro':<20} {baseline_metrics['f1_macro']:>22.4f} {aug_metrics['f1_macro']:>22.4f}")
print(f"{'Recall-Macro':<20} {baseline_metrics['recall_macro']:>22.4f} {aug_metrics['recall_macro']:>22.4f}")
print(f"{'Total Params':<20} {baseline_metrics['total_params']/1e6:>21.2f}M {aug_metrics['total_params']/1e6:>21.2f}M")
print("=" * 70)
print("\nPaper reference (MRW-CNN test accuracy on original maize/rice/wheat dataset):")
print("  Maize=0.9704, Rice=0.9706, Wheat=0.9808")
print("  NOTE: Not directly comparable — this run uses a substitute chili dataset.")
print("        Architecture (filters/kernel/order/no-pooling) is CONFIRMED from Fig. 9;")
print("        only stride=2 + padding='same' were assumed.")

## Cell 22: Install Excel Export Dependencies

In [ ]:
!pip install -q openpyxl Pillow

## Cell 23: Export results_summary.xlsx

In [ ]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

EXCEL_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "results_summary.xlsx")
MODEL_NAME        = "MRW-CNN"

HEADERS    = ["STT", "Ten Model", "Accuracy", "F1-score", "Recall",
              "Learning Curve", "Confusion Matrix", "Parameters"]
COL_WIDTHS = [6, 35, 12, 12, 12, 38, 38, 18]
IMG_ROW_HEIGHT = 200


def style_header_row(ws):
    fill   = PatternFill("solid", fgColor="1F4E79")
    font   = Font(bold=True, color="FFFFFF", size=11)
    border = Border(
        bottom=Side(style="medium", color="FFFFFF"),
        right=Side(style="thin", color="FFFFFF"),
    )
    for col, header in enumerate(HEADERS, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.fill, cell.font, cell.border = fill, font, border
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)


def write_result_row(ws, row_num, stt, metrics, lc_path, cm_path):
    data = [
        stt,
        metrics["model_name"],
        round(metrics["accuracy"], 4),
        round(metrics["f1_macro"], 4),
        round(metrics["recall_macro"], 4),
        "",
        "",
        f"{metrics['total_params']:,}",
    ]
    for col, val in enumerate(data, start=1):
        cell = ws.cell(row=row_num, column=col, value=val)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    ws.row_dimensions[row_num].height = IMG_ROW_HEIGHT

    if os.path.exists(lc_path):
        img_lc = XLImage(lc_path)
        img_lc.width, img_lc.height = 280, 190
        ws.add_image(img_lc, f"F{row_num}")

    if os.path.exists(cm_path):
        img_cm = XLImage(cm_path)
        img_cm.width, img_cm.height = 280, 190
        ws.add_image(img_cm, f"G{row_num}")


def build_sheet(wb, sheet_name, metrics, lc_path, cm_path):
    ws = wb.create_sheet(title=sheet_name)
    for col, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(col)].width = width
    ws.row_dimensions[1].height = 30
    style_header_row(ws)
    write_result_row(ws, row_num=2, stt=1, metrics=metrics,
                     lc_path=lc_path, cm_path=cm_path)
    return ws


wb = Workbook()
wb.remove(wb.active)

build_sheet(
    wb, "Baseline",
    metrics=baseline_metrics,
    lc_path=BASELINE_LEARNING_CURVE_PATH,
    cm_path=BASELINE_CONFUSION_MATRIX_PATH
)

build_sheet(
    wb, "Augmented",
    metrics=aug_metrics,
    lc_path=AUG_LEARNING_CURVE_PATH,
    cm_path=AUG_CONFUSION_MATRIX_PATH
)

wb.save(EXCEL_OUTPUT_PATH)
print(f"[Excel] Saved: {EXCEL_OUTPUT_PATH}")
print(f"  Baseline  -> Accuracy={baseline_metrics['accuracy']:.4f} | F1={baseline_metrics['f1_macro']:.4f} | Recall={baseline_metrics['recall_macro']:.4f} | Params={baseline_metrics['total_params']:,}")
print(f"  Augmented -> Accuracy={aug_metrics['accuracy']:.4f}      | F1={aug_metrics['f1_macro']:.4f}      | Recall={aug_metrics['recall_macro']:.4f}      | Params={aug_metrics['total_params']:,}")